# Guía L2 — El índice sancocho · **Sesión 2: el índice en el tiempo y el duelo de ciudades**

**Cálculo Diferencial — Laboratorio computacional** · Universidad Nacional de Colombia, Sede Manizales

Hoy la olla se pone histórica: años de precios oficiales, los picos con nombre propio, el duelo de ciudades, la apuesta de conmutatividad y la unidad monetaria más seria jamás inventada: **el sancocho**.

**Antes de ejecutar:** predicciones P4 y P5 de la guía, en papel, foto al informe. P5 es la apuesta central del laboratorio — no se la salten, que sin apuesta no hay gloria.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

RUTA = "https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/main/datos/sipsa_mensual.csv.gz"
df = pd.read_csv(RUTA)

# La receta oficial y el pipeline de la sesión 1 (los reconstruimos aquí para
# que este cuaderno sea autónomo — reproducibilidad ante todo):
RECETA = {
    "Carne de cerdo, espinazo":       1.5,
    "Yuca ICA":                       1.0,
    "Papa capira":                    1.0,
    "Chócolo mazorca":                0.9,
    "Plátano dominico hartón verde":  1.4,
}

def indice_plato(tabla, ciudad, receta):
    "tabla -> serie: costo del plato mes a mes en una ciudad. El pipeline entero, empacado."
    # Si hay varias centrales en la ciudad, se usa su promedio simple por producto y mes.
    t = tabla[(tabla["ciudad"] == ciudad) & (tabla["producto"].isin(receta))]
    p = t.pivot_table(index="fecha", columns="producto", values="precio_kg", aggfunc="mean")
    faltantes = [producto for producto in receta if producto not in p.columns]
    if faltantes:
        raise ValueError(f"{ciudad}: faltan ingredientes de la receta: {faltantes}")
    p = p[list(receta)]
    meses_incompletos = p.index[p.isna().any(axis=1)].to_list()
    if meses_incompletos:
        raise ValueError(f"{ciudad}: receta incompleta en {len(meses_incompletos)} meses; primero revisen la sustitución.")
    return p.mul(pd.Series(receta), axis="columns").sum(axis=1)

print("Pipeline listo. Fíjense: indice_plato RECIBE la tabla, la ciudad y la receta.",
      "Nada global. Cambiar de plato será cambiar un diccionario.")

---
## 1 · El índice sancocho, 2013 en adelante

Confronten con su P4: ¿la curva que bosquejaron se parece a la real?

In [ ]:
sancocho_mzl = indice_plato(df, "Manizales", RECETA)

plt.figure(figsize=(10, 4))
plt.plot(range(len(sancocho_mzl)), sancocho_mzl.values)
paso = 12
plt.xticks(range(0, len(sancocho_mzl), paso), sancocho_mzl.index[::paso], rotation=45)
plt.ylabel("Costo del sancocho ($)")
plt.title("El índice sancocho — Manizales")
plt.show()

print(f"Primer mes ({sancocho_mzl.index[0]}): ${sancocho_mzl.iloc[0]:,.0f}")
print(f"Último mes ({sancocho_mzl.index[-1]}): ${sancocho_mzl.iloc[-1]:,.0f}")
print(f"Variación total: {100*(sancocho_mzl.iloc[-1]/sancocho_mzl.iloc[0]-1):.1f}%")

🗣️ **Los picos tienen nombre.** Elijan los **dos** saltos más bruscos de la curva. Para cada uno: ¿en qué mes ocurre?, ¿qué pasó en Colombia por esas fechas? (Pistas de historiador: paros, fenómenos de El Niño o La Niña, cosechas, tasa de cambio. Busquen — con IA si quieren, documentando — y sustenten: hipótesis + evidencia, tres líneas por pico.)

*(doble clic y escriban aquí)*

---
## 2 · El duelo de ciudades (y el problema de la sustitución)

El sancocho "igual" no existe entre ciudades: el conjunto oficial obliga a decidir **equivalencias** de variedades y cortes. Además, cuando hay varias centrales en una ciudad, `indice_plato` usa el promedio simple de sus observaciones disponibles por producto y mes. El cuaderno trae una receta base que funciona los 144 meses; no es una verdad culinaria. Ambas son decisiones de modelación: defiéndanlas.

In [ ]:
# ✏️ Ejercicio 1 — Revisen las equivalencias y, si las cambian, defiendan la decisión.
# Pista: df[(df.ciudad=="Bogotá") & (df.producto.str.contains("yuca", case=False))]["producto"].unique()

RECETA_BOGOTA = {
    "Carne de cerdo, costilla":    1.5,   # sustitución del espinazo
    "Yuca llanera":               1.0,
    "Papa parda pastusa":         1.0,
    "Chócolo mazorca":            0.9,
    "Plátano hartón verde":       1.4,
}

RECETA_MEDELLIN = {
    "Carne de cerdo, espinazo":  1.5,
    "Yuca criolla":              1.0,   # sustitución de Yuca ICA
    "Papa capira":               1.0,
    "Chócolo mazorca":           0.9,
    "Plátano hartón verde":      1.4,   # sustitución del dominico hartón
}

series = {
    "Manizales": sancocho_mzl,
    "Bogotá":    indice_plato(df, "Bogotá", RECETA_BOGOTA),
    "Medellín":  indice_plato(df, "Medellín", RECETA_MEDELLIN),
}

plt.figure(figsize=(10, 4))
for ciudad, s in series.items():
    plt.plot(range(len(s)), s.values, label=f"{ciudad} (${s.iloc[-1]:,.0f})")
plt.xticks(range(0, len(sancocho_mzl), 12), sancocho_mzl.index[::12], rotation=45)
plt.ylabel("Costo del plato ($)")
plt.title("El duelo de ciudades")
plt.legend()
plt.show()

comparacion = pd.DataFrame({
    ciudad: {"inicio": serie.iloc[0], "final": serie.iloc[-1]}
    for ciudad, serie in series.items()
}).T
comparacion["aumento en pesos"] = comparacion["final"] - comparacion["inicio"]
comparacion["aumento porcentual"] = 100 * (comparacion["final"] / comparacion["inicio"] - 1)
comparacion.round({"inicio": 0, "final": 0, "aumento en pesos": 0, "aumento porcentual": 1})

🗣️ **Tabla de equivalencias (escriban aquí):** Manizales: espinazo / Yuca ICA / papa capira / dominico hartón; Bogotá: costilla / yuca llanera / parda pastusa / hartón; Medellín: espinazo / yuca criolla / capira / hartón. Defiendan en tres líneas por qué estas sustituciones permiten comparar platos sin fingir que son idénticos.

¿Dónde es más caro el almuerzo y por cuánto? ¿Las curvas suben al mismo ritmo? Y la pregunta con trampa: **"¿dónde subió más?" tiene dos respuestas legítimas** — en pesos y en porcentaje. Calculen ambas y decidan cuál reportarían, defendiendo la elección. *(escriban aquí)*

---
## 3 · La apuesta central: ¿conmutan?

P5 sobre la mesa. Dos caminos para el "costo típico anual" del sancocho:

- **Camino A:** promediar los 12 precios del año de cada ingrediente → costear la receta una vez.
- **Camino B:** costear la receta mes a mes → promediar los 12 costos.

¿Dan lo mismo? Primero su apuesta escrita, luego el juez:

In [ ]:
AÑO = "2016"
mzl = df[(df["ciudad"] == "Manizales") & (df["producto"].isin(RECETA))]
anual = mzl[mzl["fecha"].str.startswith(AÑO)]

# Camino A: promediar precios, luego costear
precios_promedio = anual.groupby("producto")["precio_kg"].mean()
camino_A = sum(RECETA[p] * precios_promedio[p] for p in RECETA)

# Camino B: costear cada mes, luego promediar
camino_B = indice_plato(df, "Manizales", RECETA)
camino_B = camino_B[camino_B.index.str.startswith(AÑO)].mean()

print(f"Camino A (promediar → costear): ${camino_A:,.2f}")
print(f"Camino B (costear → promediar): ${camino_B:,.2f}")
print(f"Diferencia: ${abs(camino_A - camino_B):.6f}")

In [ ]:
# ✏️ Ejercicio 2 — Rompan la magia.
# Repitan los dos caminos usando la MEDIANA en lugar del promedio.
# ¿Siguen dando lo mismo? ¿Y si antes de promediar eliminan el 5% de precios más caros?

# camino_A_mediana = ...
# camino_B_mediana = ...
# print(...)

# 🗣️ Moraleja (escríbanla como comentario): los dos caminos coinciden con el promedio
# porque la receta es __________ y el promedio también lo es; con la mediana se rompe
# porque __________.

---
## 4 · El camino de regreso: la unidad sancocho

Del costo total del plato no se puede recuperar el precio del plátano — la suma colapsa infinitas combinaciones en un número (no inyectiva: información que muere). Pero hay una conversión que sí tiene camino de regreso perfecto:

In [ ]:
VALOR_SANCOCHO = float(sancocho_mzl.iloc[-1])   # 1 sancocho = costo del último mes del congelado

def a_sancochos(pesos):
    return pesos / VALOR_SANCOCHO

def a_pesos(sancochos):
    return sancochos * VALOR_SANCOCHO

# La prueba de fuego de una inversa: ida y vuelta = quedarse quieto.
print("¿a_pesos(a_sancochos(x)) == x?  →", a_pesos(a_sancochos(123456.0)))

# ✏️ Ejercicio 3 — Conviertan a sancochos: el salario mínimo de ese año, la matrícula,
# y la mesada (o el presupuesto mensual de comida) de alguien del equipo que quiera confesarla.
SALARIO_MINIMO = 0        # ← búsquenlo (¡documenten la fuente!)
# print(f"El salario mínimo equivale a {a_sancochos(SALARIO_MINIMO):.1f} sancochos")

# 🗣️ ¿Y la pregunta incómoda: el salario mínimo de 2013 alcanzaba para más o menos
# sancochos que el del último año del dataset? Calcúlenlo. Eso — poder adquisitivo
# medido en almuerzos — es economía seria con matemáticas de primer semestre.

---
## 5 · El plato propio (proyecto del equipo)

Cambiar de plato es cambiar un diccionario — esa es la recompensa de haber escrito el pipeline como funciones. Elijan SU plato (ajiaco, mondongo, frijolada, o la receta de su casa con las cantidades que les dicte quien cocina) y construyan su índice.

In [ ]:
# ✏️ Ejercicio 4 — Su plato, su índice.
RECETA_PROPIA = {
    # "Producto oficial del DANE": kilos,
}
NOMBRE_PLATO = "———"
CIUDAD_PROPIA = "Manizales"

if RECETA_PROPIA:
    mi_indice = indice_plato(df, CIUDAD_PROPIA, RECETA_PROPIA)
    plt.figure(figsize=(10, 4))
    plt.plot(range(len(mi_indice)), mi_indice.values, label=NOMBRE_PLATO)
    plt.plot(range(len(sancocho_mzl)), sancocho_mzl.values, label="Sancocho (referencia)")
    plt.xticks(range(0, len(sancocho_mzl), 12), sancocho_mzl.index[::12], rotation=45)
    plt.ylabel("Costo del plato ($)")
    plt.legend()
    plt.title(f"El índice {NOMBRE_PLATO}")
    plt.show()
    # ¿Su plato se infló más o menos que el sancocho? ¿En qué años se separan las curvas?

---
## 6 · Momento 3 — Explicar + la cacería estrella

🗣️ Una página del equipo: (1) la apuesta de conmutatividad — qué apostaron, qué dio el juez, y la moraleja con la propiedad nombrada; (2) la clasificación de invertibilidad de su pipeline (¿qué pasos tienen camino de regreso?); (3) la lectura histórica de sus dos picos; (4) el veredicto del duelo de ciudades con su tabla de equivalencias defendida.

🤖 **Y el cierre de la cacería estrella:** en la sesión 1 le preguntaron a su IA cuánto cuesta un sancocho para seis. Peguen aquí su respuesta de entonces y el número de SU índice para el mes actual del congelado. ¿Quién tenía la cifra y quién tenía el cuento? Al expediente (bitácora individual) — esta comparación es la demostración empírica de la regla del curso: *la IA no es la autoridad final; los datos, sí.*

**Entregables:** cuaderno ejecutado completo + bitácora individual + tarea puente (verificación reproducida y defensa del precio medio). Rúbrica en la guía. Cuaderno que no ejecuta: se devuelve sin calificar.